# ADK Workshop - Environment Verification

## Purpose

This notebook validates your development environment before the workshop. Running these checks **48 hours before the workshop** ensures you catch authentication, dependency, and API access issues early - preventing setup delays during the session.

## Confirmation Required

After running this notebook:
1. **Screenshot** the "READY FOR WORKSHOP" message
2. **Email** the screenshot to your instructor
3. **Wait** for instructor confirmation (within 24 hours)

The instructor will confirm your environment is ready and you're registered for the workshop.

## Prerequisites

**You need a Google AI API Key.** Get one free at: https://aistudio.google.com/apikey

## Instructions

1. **Run all cells** in order by clicking Runtime -> Run all (or Ctrl+F9)
2. **Enter your API key** when prompted (it will be hidden)
3. **Look for green checkmarks** indicating successful verification
4. **Confirm success** by seeing "READY FOR WORKSHOP" at the end

**Note:** Complete this verification at least 48 hours before the workshop to allow time for troubleshooting if needed.

In [ ]:
# Cell 1: Install Dependencies
# Install Google Agent Development Kit

!pip install -q google-adk==1.23.0

print("google-adk 1.23.0 installed successfully")

In [ ]:
# Cell 2: Configure API Key
# Securely enter your Google AI API Key

import os
import google.generativeai as genai
from getpass import getpass

# Prompt for API key (input will be hidden)
api_key = getpass('Enter your Google AI API Key: ')

# Configure the generative AI library with the provided key
genai.configure(api_key=api_key)

# Set as environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

print("API Key configured successfully!")

In [ ]:
# Cell 3: Verify Dependencies
# Check Python version and ADK installation

import sys

print("Dependency Verification:")
print("=" * 50)

# Check Python version
python_version = sys.version_info
if python_version >= (3, 11):
    print(f"PASS: Python {python_version.major}.{python_version.minor}.{python_version.micro}")
else:
    print(f"FAIL: Python {python_version.major}.{python_version.minor} (requires 3.11+)")

# Check ADK version
try:
    import google.adk
    adk_version = google.adk.__version__
    if adk_version == '1.23.0':
        print(f"PASS: google-adk {adk_version}")
    else:
        print(f"WARNING: google-adk {adk_version} (expected 1.23.0)")
except ImportError:
    print("FAIL: google-adk not installed")
    print("  -> Run: !pip install google-adk==1.23.0")

# Check API key is set
if os.environ.get('GOOGLE_API_KEY'):
    print("PASS: GOOGLE_API_KEY is set")
else:
    print("FAIL: GOOGLE_API_KEY not set")
    print("  -> Go back and run Cell 2")

print("=" * 50)

In [ ]:
# Cell 4: Test ADK Agent
# This is the key test - verify ADK can create and run agents

print("ADK Agent Test:")
print("=" * 50)

try:
    from google.adk.agents import Agent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai.types import Content, Part

    print("Creating test agent...")

    # Create a simple test agent (uses GOOGLE_API_KEY automatically)
    test_agent = Agent(
        model='gemini-2.5-flash',
        name='verification_agent',
        description='Simple agent for environment verification.',
        instruction='You are a test agent. Respond with exactly: "Hello! Setup verified."',
    )

    # Set up session service and runner
    session_service = InMemorySessionService()
    runner = Runner(
        agent=test_agent,
        session_service=session_service,
        app_name='verification_test'
    )

    # Test the agent
    async def test_adk_agent():
        session = await session_service.create_session(
            app_name='verification_test',
            user_id='test_user'
        )

        final_response = ""
        async for event in runner.run_async(
            user_id='test_user',
            session_id=session.id,
            new_message=Content(parts=[Part(text="Say hello")], role="user")
        ):
            if event.is_final_response():
                final_response = event.content.parts[0].text
                break

        return final_response

    # Run the test
    response = await test_adk_agent()

    if response and len(response) > 0:
        print("PASS: ADK Agent working!")
        preview = response[:100].replace('\n', ' ')
        print(f"  Agent response: {preview}...")
    else:
        print("FAIL: Empty response from agent")

except Exception as e:
    print(f"FAIL: {str(e)}")
    print("\nTROUBLESHOOTING:")
    print("  -> Verify your API key is correct")
    print("  -> Get a new key at: https://aistudio.google.com/apikey")
    print("  -> Rerun Cell 2 with the correct key")

print("=" * 50)

In [ ]:
# Cell 5: Final Verification Summary
# Run all checks and show consolidated report

import sys

print("\n" + "=" * 60)
print("ENVIRONMENT VERIFICATION SUMMARY")
print("=" * 60)

checks_passed = 0
checks_failed = 0

# Check 1: Python version
python_version = sys.version_info
if python_version >= (3, 11):
    print(f"PASS: Python {python_version.major}.{python_version.minor}")
    checks_passed += 1
else:
    print(f"FAIL: Python {python_version.major}.{python_version.minor} (need 3.11+)")
    checks_failed += 1

# Check 2: ADK installed
try:
    import google.adk
    print(f"PASS: google-adk {google.adk.__version__}")
    checks_passed += 1
except ImportError:
    print("FAIL: google-adk not installed")
    checks_failed += 1

# Check 3: API key set
if os.environ.get('GOOGLE_API_KEY'):
    print("PASS: API key configured")
    checks_passed += 1
else:
    print("FAIL: API key not configured")
    checks_failed += 1

# Check 4: Agent works (quick test)
try:
    from google.adk.agents import Agent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai.types import Content, Part
    
    agent = Agent(
        model='gemini-2.5-flash',
        name='quick_test',
        description='Quick test',
        instruction='Say OK',
    )
    
    svc = InMemorySessionService()
    runner = Runner(agent=agent, session_service=svc, app_name='quick_test')
    
    async def quick_test():
        session = await svc.create_session(app_name='quick_test', user_id='test')
        async for event in runner.run_async(
            user_id='test',
            session_id=session.id,
            new_message=Content(parts=[Part(text="Hi")], role="user")
        ):
            if event.is_final_response():
                return event.content.parts[0].text
        return ""
    
    result = await quick_test()
    if result:
        print("PASS: ADK Agent responding")
        checks_passed += 1
    else:
        print("FAIL: ADK Agent not responding")
        checks_failed += 1
except Exception as e:
    print(f"FAIL: ADK Agent error - {str(e)[:50]}")
    checks_failed += 1

print("=" * 60)
print(f"Results: {checks_passed} passed, {checks_failed} failed")
print("=" * 60)

if checks_failed == 0:
    print("\nREADY FOR WORKSHOP!")
    print("All checks passed. You're good to go!")
    print("\nYour environment:")
    print("  - Python 3.11+")
    print("  - Google ADK installed")
    print("  - API key configured")
    print("  - Agent responding")
    print("\n" + "=" * 60)
    print("CONFIRMATION REQUIRED")
    print("=" * 60)
    print("\nTo confirm your environment is ready:")
    print("")
    print("1. Take a screenshot of this 'READY FOR WORKSHOP' message")
    print("2. Email the screenshot to your instructor")
    print("   Subject: '[Workshop Name] Environment Verified - [Your Name]'")
    print("3. You'll receive confirmation within 24 hours")
    print("")
    print("If you don't receive confirmation, contact the instructor.")
    print("=" * 60)
else:
    print("\nNOT READY - Please fix the failed checks above")
    print("\nFull troubleshooting guide: See TROUBLESHOOTING.md")
    print("Still stuck? Email instructor with error screenshot")
    print("\nQuick fixes:")
    print("  -> Get API key: https://aistudio.google.com/apikey")
    print("  -> Reinstall ADK: !pip install google-adk==1.23.0")

## Next Steps

### All Checks Passed

**Congratulations! You're ready for the workshop!**

Your environment is properly configured with:
- Python 3.11+
- Google ADK 1.23.0
- Google AI API key configured
- ADK Agent responding

### Important: Confirm Your Setup

Don't forget to:
1. Screenshot the "READY FOR WORKSHOP" message above
2. Email it to your instructor
3. Subject line: "[Workshop Name] Environment Verified - [Your Name]"

You'll receive confirmation within 24 hours.

---

### Some Checks Failed

**Don't worry! Here's how to fix common issues:**

#### "API key not configured"
1. Get a free API key at: https://aistudio.google.com/apikey
2. Re-run Cell 2 and enter your key

#### "ADK not installed"
Run: `!pip install google-adk==1.23.0`

#### "Agent not responding"
- Check your API key is valid
- Try getting a new key from https://aistudio.google.com/apikey
- Make sure you have internet connectivity

**Need more help?** See TROUBLESHOOTING.md for comprehensive solutions.

---

## What's Next?

Continue to **Exercise 1: Your First ADK Agent** to start building!